In [9]:
%pip install opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 MB 72.9 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path
import os
import cv2
import numpy as np
import pandas as pd


USER = "meryv"

video_folder = Path("/Users") / USER / "Desktop" / "sleap" / "clean"

perimeters_folder = (
    Path("/Users") / USER / "repos" / "sensory-deprivation" / "perimeters"
)
output_folder = perimeters_folder

perimeter_centres_file = output_folder / "perimeter_centres.csv"


def detect_largest_circle(frame):
    """
    Detect the largest circular object in the frame.
    Returns x, y, radius in pixels.
    """

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray_blurred = cv2.medianBlur(gray, 5)

    circles = cv2.HoughCircles(
        gray_blurred,
        cv2.HOUGH_GRADIENT,
        dp=1.0,
        minDist=100,
        param1=500,
        param2=50,
        minRadius=400,
        maxRadius=600,
    )

    if circles is None:
        return None

    largest_circle = max(circles[0, :], key=lambda c: c[2])

    return largest_circle


centre_rows = []

video_files = sorted(video_folder.glob("*.mp4"))



for video_path in video_files:

    video_file = video_path.name
    video_stem = video_path.stem

    expected_feather_file = f"{video_stem}.tracks.feather"

    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, 10)

    ret, frame = cap.read()
    cap.release()

    if not ret:
        print(f"Could not read frame 10 from {video_name}")

        centre_rows.append({
            "video_name": video_name,
            "centre_x": np.nan,
            "centre_y": np.nan,
            "radius": np.nan,
            "diameter": np.nan,
            "mm_per_pixel": np.nan,
            "perimeter_detected": False,
        })

        continue

    circle = detect_largest_circle(frame)

    if circle is None:
        print(f"No perimeter detected for {video_name}")

        centre_rows.append({
            "video_name": video_name,
            "centre_x": np.nan,
            "centre_y": np.nan,
            "radius": np.nan,
            "diameter": np.nan,
            "mm_per_pixel": np.nan,
            "perimeter_detected": False,
        })

        continue

    centre_x, centre_y, radius = circle

    diameter = radius * 2
    mm_per_pixel = 90 / diameter

    centre_rows.append({
        "video_name": video_name,
        "centre_x": float(centre_x),
        "centre_y": float(centre_y),
        "radius": float(radius),
        "diameter": float(diameter),
        "mm_per_pixel": float(mm_per_pixel),
        "perimeter_detected": True,
    })

    # Save QC image
    qc_frame = frame.copy()

    cv2.circle(
        qc_frame,
        (int(centre_x), int(centre_y)),
        int(radius),
        (0, 255, 0),
        2,
    )

    cv2.circle(
        qc_frame,
        (int(centre_x), int(centre_y)),
        5,
        (0, 0, 255),
        -1,
    )

    qc_path = output_folder / f"{video_name}_perimeter_qc.png"
    cv2.imwrite(str(qc_path), qc_frame)

    print(
        f"{video_name}: centre=({centre_x:.2f}, {centre_y:.2f}), "
        f"diameter={diameter:.2f}px, mm_per_pixel={mm_per_pixel:.5f}"
    )


centre_df = pd.DataFrame(centre_rows)

centre_df.to_csv(perimeter_centres_file, index=False)

print(f"\nSaved: {perimeter_centres_file}")

centre_df

Found 13 video files.
2026-03-20_11-36-47_td3_23129: centre=(731.50, 701.50), diameter=1039.00px, mm_per_pixel=0.08662
2026-03-20_12-15-57_td18_W1118: centre=(679.50, 726.50), diameter=1031.20px, mm_per_pixel=0.08728
2026-03-20_12-17-07_td19_W1118: centre=(705.50, 688.50), diameter=961.20px, mm_per_pixel=0.09363
2026-03-23_11-38-40_td2_23129: centre=(708.50, 727.50), diameter=1026.60px, mm_per_pixel=0.08767
2026-03-23_11-45-39_td5_33300: centre=(716.50, 707.50), diameter=1030.00px, mm_per_pixel=0.08738
2026-03-23_12-18-46_td16_33300: centre=(749.50, 720.50), diameter=1047.20px, mm_per_pixel=0.08594
2026-03-31_11-44-00_td3_9047: centre=(734.50, 697.50), diameter=1037.80px, mm_per_pixel=0.08672
2026-03-31_11-47-00_td4_23129: centre=(747.50, 739.50), diameter=970.00px, mm_per_pixel=0.09278
2026-03-31_11-53-03_td6_33300: centre=(706.50, 728.50), diameter=1032.40px, mm_per_pixel=0.08718
2026-03-31_12-00-13_td9_W1118: centre=(722.50, 689.50), diameter=1034.60px, mm_per_pixel=0.08699
2026-03-

,video_name,centre_x,centre_y,radius,diameter,mm_per_pixel,perimeter_detected
0,2026-03-20_11-36-47_td3_23129,731.5,701.5,519.500000,1039.000000,0.086622,True
1,2026-03-20_12-15-57_td18_W1118,679.5,726.5,515.599976,1031.199951,0.087277,True
2,2026-03-20_12-17-07_td19_W1118,705.5,688.5,480.600006,961.200012,0.093633,True
3,2026-03-23_11-38-40_td2_23129,708.5,727.5,513.299988,1026.599976,0.087668,True
4,2026-03-23_11-45-39_td5_33300,716.5,707.5,515.000000,1030.000000,0.087379,True
5,2026-03-23_12-18-46_td16_33300,749.5,720.5,523.599976,1047.199951,0.085943,True
6,2026-03-31_11-44-00_td3_9047,734.5,697.5,518.900024,1037.800049,0.086722,True
7,2026-03-31_11-47-00_td4_23129,747.5,739.5,485.000000,970.000000,0.092784,True
8,2026-03-31_11-53-03_td6_33300,706.5,728.5,516.200012,1032.400024,0.087176,True
9,2026-03-31_12-00-13_td9_W1118,722.5,689.5,517.299988,1034.599976,0.086990,True
